In [2]:
%pip install torch transformers pandas

Note: you may need to restart the kernel to use updated packages.


필수 라이브러리 로드 및 환경 세팅

In [3]:
# 필요한 라이브러리 불러오기
import torch
import pandas as pd
from transformers import BertTokenizer, BertForSequenceClassification
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW

# GPU(CUDA) 또는 CPU 설정 (애플 실리콘 Mac의 경우 'mps', 윈도우/리눅스는 'cuda')
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"🚀 현재 사용 중인 디바이스: {device}")

/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🚀 현재 사용 중인 디바이스: mps


실제 데이터(ratings_train.txt) 불러오기

In [4]:
# 1. 텍스트 파일 불러오기 (탭 \t 으로 구분되어 있음)
df = pd.read_csv('ratings_train.txt', sep='\t')

# 2. 비어있는(NaN) 데이터 제거 (전처리의 가장 기본)
df = df.dropna()

print(f"✅ 전체 데이터 개수: {len(df)}개")

# 🚨 중요: 15만 개를 처음부터 다 돌리면 몇 시간이 걸립니다.
# 먼저 코드가 잘 돌아가는지 확인하기 위해 1,000개만 샘플링해서 뼈대를 테스트합니다.
# 나중에 최종 학습할 때는 이 아래 줄을 지우거나 주석(#) 처리하세요!
df_sample = df.sample(n=1000, random_state=42).reset_index(drop=True)

print("미리보기:")
display(df_sample.head())

✅ 전체 데이터 개수: 149995개
미리보기:


,id,document,label
0,7865795,원본이 최고,1
1,5417631,스릴감과 훈훈함이 있는 영화.,1
2,8357466,굉장히 저평가되는 영화중 하나라고 생각함,1
3,8252946,정말영화같은이야기 영화여서 영화같은이야기가 좋다,1
4,7800452,계기도없는데 이상하다,0


토크나이저 및 PyTorch 데이터셋 세팅. Tokenizer(사람의 말을 AI가 이해할 수 있는 숫자 암호로 번역해 주는 사전)만들고 모델에 넣을 준비

In [5]:
# 3. 모델 이름: 한국어 구어체에 강력한 최신 KcBERT 사용
model_name = 'beomi/kcbert-base' 
tokenizer = BertTokenizer.from_pretrained(model_name)

# 4. PyTorch Dataset 클래스 정의 (최신 문법 적용)
class NsmcDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=64):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        text = str(self.data['document'].iloc[index])
        label = self.data['label'].iloc[index]

        # 🚨 encode_plus 대신 최신 트렌드인 직접 호출(__call__) 방식 사용
        inputs = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': inputs['input_ids'].flatten(),
            'attention_mask': inputs['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# 5. DataLoader 생성
train_dataset = NsmcDataset(df_sample, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
print("✅ 토크나이징 및 DataLoader 세팅 완료!")

✅ 토크나이징 및 DataLoader 세팅 완료!


KcBERT 모델 로드 및 학습(Train) 진행
뼈대만 있는 KoBERT 모델을 가져와서, 우리가 만든 데이터를 총 3번(epochs=3) 반복해서 읽히면서 "이 단어 조합은 악플(0)이야, 이건 칭찬(1)이야" 하고 뇌를 훈련(Fine-tuning)시키기

In [6]:
# 6. 분류용 모델 로드 (긍정/부정 2가지 레이블)
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=2)
model.to(device)

# 7. 옵티마이저 설정
optimizer = AdamW(model.parameters(), lr=2e-5)
epochs = 3 # 데이터를 총 3번 반복해서 봅니다.

print("🔥 딥러닝 모델 학습 시작!")

# 8. 학습 루프
for epoch in range(epochs):
    model.train()
    total_loss = 0
    
    for batch_idx, batch in enumerate(train_loader):
        optimizer.zero_grad()
        
        # 데이터를 장치(GPU 등)로 이동
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        # 모델 예측 및 오차(Loss) 계산
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        
        # 역전파(Backpropagation) 및 가중치 업데이트
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        # 중간 경과 출력 (10번째 배치마다)
        if batch_idx % 10 == 0:
            print(f"Epoch [{epoch+1}/{epochs}] | Batch [{batch_idx}/{len(train_loader)}] | Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(train_loader)
    print(f"🎯 Epoch {epoch+1} 완료! 평균 Loss: {avg_loss:.4f}\n")

print("🎉 샘플 데이터 학습이 완벽하게 끝났습니다!")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7043.48it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: beomi/kcbert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

🔥 딥러닝 모델 학습 시작!
Epoch [1/3] | Batch [0/63] | Loss: 0.6876
Epoch [1/3] | Batch [10/63] | Loss: 0.6089
Epoch [1/3] | Batch [20/63] | Loss: 0.6776
Epoch [1/3] | Batch [30/63] | Loss: 0.3988
Epoch [1/3] | Batch [40/63] | Loss: 0.2869
Epoch [1/3] | Batch [50/63] | Loss: 0.4848
Epoch [1/3] | Batch [60/63] | Loss: 0.3919
🎯 Epoch 1 완료! 평균 Loss: 0.5548

Epoch [2/3] | Batch [0/63] | Loss: 0.4754
Epoch [2/3] | Batch [10/63] | Loss: 0.4976
Epoch [2/3] | Batch [20/63] | Loss: 0.4498
Epoch [2/3] | Batch [30/63] | Loss: 0.0605
Epoch [2/3] | Batch [40/63] | Loss: 0.2707
Epoch [2/3] | Batch [50/63] | Loss: 0.5028
Epoch [2/3] | Batch [60/63] | Loss: 0.2569
🎯 Epoch 2 완료! 평균 Loss: 0.3094

Epoch [3/3] | Batch [0/63] | Loss: 0.1103
Epoch [3/3] | Batch [10/63] | Loss: 0.0390
Epoch [3/3] | Batch [20/63] | Loss: 0.0199
Epoch [3/3] | Batch [30/63] | Loss: 0.0341
Epoch [3/3] | Batch [40/63] | Loss: 0.0297
Epoch [3/3] | Batch [50/63] | Loss: 0.0132
Epoch [3/3] | Batch [60/63] | Loss: 0.1538
🎯 Epoch 3 완료! 평균 Loss: